In [72]:
import pandas as pd
import numpy as np

import mysql.connector
from config import DB_CONFIG

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report , confusion_matrix

In [2]:
conn = mysql.connector.connect(**DB_CONFIG)

if conn.is_connected():
    print("Database Connected Successfully!")

Database Connected Successfully!


In [3]:
df = pd.read_sql("select * from ecommerce_cleaned", con = conn)
conn.close()

C:\Users\Ekamjot Singh\AppData\Local\Temp\ipykernel_23100\1814627627.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("select * from ecommerce_cleaned", con = conn)


In [4]:
df.shape

(5628, 20)

In [25]:
df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferredOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4,Mobile Phone,3,6,Debit Card,Female,3,3,Laptop & Accessory,2,Single,9,1,11,1,1,5,159.93
1,50002,1,1,Mobile Phone,1,8,UPI,Male,3,4,Mobile Phone,3,Single,7,1,15,0,1,0,120.90
2,50003,1,1,Mobile Phone,1,30,Debit Card,Male,2,4,Mobile Phone,3,Single,6,1,14,0,1,3,120.28
3,50004,1,0,Mobile Phone,3,15,Debit Card,Male,2,4,Laptop & Accessory,5,Single,8,0,23,0,1,3,134.07
4,50005,1,0,Mobile Phone,1,12,Credit Card,Male,3,3,Mobile Phone,5,Single,3,0,11,1,1,3,129.60


In [5]:
df.dtypes

CustomerID                       int64
Churn                            int64
Tenure                           int64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                  int64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                   int64
NumberOfDeviceRegistered         int64
PreferredOrderCat               object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear      int64
CouponUsed                       int64
OrderCount                       int64
DaySinceLastOrder                int64
CashbackAmount                 float64
dtype: object

In [112]:
categorical_cols = ['PreferredLoginDevice', 'CityTier', 'PreferredPaymentMode', 'Gender',
                    'PreferredOrderCat', 'SatisfactionScore',
                    'MaritalStatus']

for col in categorical_cols:
    df[col] = df[col].astype('category')

In [7]:
customer_ids = df["CustomerID"]
df_model = df.drop(columns = ["CustomerID"])

In [10]:
df_model = df_model.drop(columns=['OrderAmountHikeFromlastYear'])    # weakest feature removed

print(df_model.shape)
print(df_model.columns.tolist())

(5628, 18)
['Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferredOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


In [12]:
# Separate Features and Target

X = df_model.drop(columns = ["Churn"])
y = df_model["Churn"]

In [14]:
print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

Features shape: (5628, 17)
Target distribution:
 Churn
0    0.831557
1    0.168443
Name: proportion, dtype: float64


In [19]:
nominal_cols = ["PreferredLoginDevice","PreferredPaymentMode","Gender","PreferredOrderCat","MaritalStatus"]
ordinal_cols = ["CityTier","SatisfactionScore"]
numeric_cols = ["Tenure","WarehouseToHome","HourSpendOnApp","NumberOfDeviceRegistered","NumberOfAddress","CouponUsed","OrderCount","DaySinceLastOrder","CashbackAmount"]
binary_cols = ['Complain']

print("Nominal:", nominal_cols)
print("Ordinal:", ordinal_cols)
print("Numeric:", numeric_cols)
print("Binary:", binary_cols)

total = len(nominal_cols) + len(ordinal_cols) + len(numeric_cols) + len(binary_cols)
print(f"\nTotal grouped: {total}")
print(f"X columns: {X.shape[1]}")
print("Match:", total == X.shape[1])

Nominal: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferredOrderCat', 'MaritalStatus']
Ordinal: ['CityTier', 'SatisfactionScore']
Numeric: ['Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'NumberOfAddress', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
Binary: ['Complain']

Total grouped: 17
X columns: 17
Match: True


In [21]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2, stratify = y, random_state = 42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain churn ratio:\n", y_train.value_counts(normalize=True))
print("\nTest churn ratio:\n", y_test.value_counts(normalize=True))

Train shape: (4502, 17)
Test shape: (1126, 17)

Train churn ratio:
 Churn
0    0.83163
1    0.16837
Name: proportion, dtype: float64

Test churn ratio:
 Churn
0    0.831261
1    0.168739
Name: proportion, dtype: float64


In [54]:
preprocessor = ColumnTransformer(
    transformers = [
        ('nominal', OneHotEncoder(drop = "first", handle_unknown = "ignore"), nominal_cols),
        ('ordinal', StandardScaler() , ordinal_cols),
        ('numeric', StandardScaler() , numeric_cols),
        ('binary', "passthrough" , binary_cols)
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers = [
        ('nominal', OneHotEncoder(drop = "first", handle_unknown = "ignore"), nominal_cols),
        ('ordinal', "passthrough", ordinal_cols),
        ('numeric', "passthrough", numeric_cols),
        ('binary', "passthrough", binary_cols)
    ]
)

In [43]:
log_reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(class_weight = "balanced",max_iter = 1000, random_state = 42))
])

log_reg_pipeline.fit(X_train, y_train)

print("Pipeline fitted successfully.")

Pipeline fitted successfully.


In [44]:
print("Train accuracy:", round(log_reg_pipeline.score(X_train, y_train),4)*100,"%")
print("Test accuracy:", round(log_reg_pipeline.score(X_test, y_test),4)*100,"%")

Train accuracy: 82.87 %
Test accuracy: 82.33 %


In [46]:
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = cross_validate(log_reg_pipeline,X_train,y_train,scoring = scoring,cv = skf, return_train_score = True)

print("Logistic Regression — 5-Fold Cross-Validation Results\n")
for metric in scoring:
    train_scores = cv_results[f'train_{metric}']
    test_scores = cv_results[f'test_{metric}']
    print(f"{metric.upper():10s} | Train: {train_scores.mean():.3f} (+/- {train_scores.std():.3f})"
          f"  | Val: {test_scores.mean():.3f} (+/- {test_scores.std():.3f})")

Logistic Regression — 5-Fold Cross-Validation Results

ACCURACY   | Train: 0.825 (+/- 0.005)  | Val: 0.821 (+/- 0.015)
PRECISION  | Train: 0.489 (+/- 0.008)  | Val: 0.483 (+/- 0.024)
RECALL     | Train: 0.850 (+/- 0.008)  | Val: 0.836 (+/- 0.018)
F1         | Train: 0.621 (+/- 0.008)  | Val: 0.612 (+/- 0.020)
ROC_AUC    | Train: 0.907 (+/- 0.003)  | Val: 0.902 (+/- 0.011)


In [53]:
# log_reg_pipeline was already fit on the full X_train earlier
y_test_pred = log_reg_pipeline.predict(X_test)
y_test_proba = log_reg_pipeline.predict_proba(X_test)[:,1]

test_scores = {
    'accuracy' : accuracy_score(y_test, y_test_pred),
    'precision' : precision_score(y_test, y_test_pred),
    'recall': recall_score(y_test, y_test_pred),
    'f1' : f1_score(y_test, y_test_pred),
    'roc_auc' : roc_auc_score(y_test, y_test_proba)
}

print("Logistic Regression — Held-Out Test Set Performance\n")
for metric, score in test_scores.items():
    print(f"{metric.upper():10s} | Test: {score:.3f}")

print("\n--- Comparison: Cross-Validation (mean) vs Test ---")
for metric in scoring:
    cv_val_mean = cv_results[f'test_{metric}'].mean()
    print(f"{metric.upper():10s} | CV Val: {cv_val_mean:.3f}  | Test: {test_scores[metric]:.3f}"
          f"  | Diff: {test_scores[metric] - cv_val_mean:+.3f}")

Logistic Regression — Held-Out Test Set Performance

ACCURACY   | Test: 0.823
PRECISION  | Test: 0.486
RECALL     | Test: 0.847
F1         | Test: 0.618
ROC_AUC    | Test: 0.902

--- Comparison: Cross-Validation (mean) vs Test ---
ACCURACY   | CV Val: 0.821  | Test: 0.823  | Diff: +0.003
PRECISION  | CV Val: 0.483  | Test: 0.486  | Diff: +0.004
RECALL     | CV Val: 0.836  | Test: 0.847  | Diff: +0.011
F1         | CV Val: 0.612  | Test: 0.618  | Diff: +0.006
ROC_AUC    | CV Val: 0.902  | Test: 0.902  | Diff: +0.001


In [59]:
rf_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestClassifier(n_estimators=200,max_depth=10,min_samples_split=10,min_samples_leaf=5,class_weight = "balanced",random_state = 42))
])

rf_pipeline.fit(X_train, y_train)

print("Random Forest pipeline fitted.")
print("Train accuracy:", rf_pipeline.score(X_train, y_train))
print("Test accuracy:", rf_pipeline.score(X_test, y_test))

Random Forest pipeline fitted.
Train accuracy: 0.9533540648600622
Test accuracy: 0.9404973357015985


In [60]:
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
scoring = ["accuracy","precision","recall","f1","roc_auc"]
rf_cv_results = cross_validate(rf_pipeline, X_train, y_train, scoring = scoring, cv = skf, return_train_score = True)

print("Random Forest Classifier — 5-Fold Cross-Validation Results\n")

for metric in scoring:
    train_scores = rf_cv_results[f'train_{metric}']
    val_scores = rf_cv_results[f'test_{metric}']
    print(f"{metric.upper():10s} | Train: {train_scores.mean():.3f} (+/- {train_scores.std():.3f})"
          f"  | Val: {val_scores.mean():.3f} (+/- {val_scores.std():.3f})")

Random Forest Classifier — 5-Fold Cross-Validation Results

ACCURACY   | Train: 0.953 (+/- 0.002)  | Val: 0.918 (+/- 0.005)
PRECISION  | Train: 0.804 (+/- 0.006)  | Val: 0.724 (+/- 0.019)
RECALL     | Train: 0.950 (+/- 0.004)  | Val: 0.827 (+/- 0.011)
F1         | Train: 0.871 (+/- 0.004)  | Val: 0.772 (+/- 0.009)
ROC_AUC    | Train: 0.991 (+/- 0.001)  | Val: 0.962 (+/- 0.002)


In [68]:
rf_y_pred = rf_pipeline.predict(X_test)
rf_y_proba = rf_pipeline.predict_proba(X_test)[:,1]

rf_test_scores = {
    'accuracy': accuracy_score(y_test, rf_y_pred),
    'precision': precision_score(y_test, rf_y_pred),
    'recall' : recall_score(y_test, rf_y_pred),
    'f1' : f1_score(y_test, rf_y_pred),
    'roc_auc' : roc_auc_score(y_test, rf_y_proba)
}

print("Random Forest Classifier — Held-Out Test Set Performance\n")
for metric, score in rf_test_scores.items():
    print(f"{metric.upper():10s} | Test: {score:.3f}")

print("\n--- Comparison: Cross-Validation (mean) vs Test ---")
for metric in scoring:
    cv_val_mean = rf_cv_results[f"test_{metric}"].mean()
    print(f"{metric.upper():10s} | Cv Val: {cv_val_mean:.3f} | Test: {rf_test_scores[metric]:.3f} | Diff: {rf_test_scores[metric] - cv_val_mean:+.3f}")

print("\nClassification Report:\n")
print(classification_report(y_test, rf_y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_y_pred))

Random Forest Classifier — Held-Out Test Set Performance

ACCURACY   | Test: 0.940
PRECISION  | Test: 0.800
RECALL     | Test: 0.863
F1         | Test: 0.830
ROC_AUC    | Test: 0.974

--- Comparison: Cross-Validation (mean) vs Test ---
ACCURACY   | Cv Val: 0.918 | Test: 0.940 | Diff: +0.023
PRECISION  | Cv Val: 0.724 | Test: 0.800 | Diff: +0.076
RECALL     | Cv Val: 0.827 | Test: 0.863 | Diff: +0.036
F1         | Cv Val: 0.772 | Test: 0.830 | Diff: +0.059
ROC_AUC    | Cv Val: 0.962 | Test: 0.974 | Diff: +0.012

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.96      0.96       936
           1       0.80      0.86      0.83       190

    accuracy                           0.94      1126
   macro avg       0.89      0.91      0.90      1126
weighted avg       0.94      0.94      0.94      1126

Confusion Matrix:
[[895  41]
 [ 26 164]]


In [73]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 4.939313984168866


In [92]:
xgb_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", XGBClassifier(n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42))
])

xgb_pipeline.fit(X_train, y_train)

print("XGBoost Pipeline fitted.")
print("Train Accuracy :",xgb_pipeline.score(X_train, y_train))
print("Test Accuracy :", xgb_pipeline.score(X_test, y_test))

XGBoost Pipeline fitted.
Train Accuracy : 0.998223011994669
Test Accuracy : 0.9875666074600356


In [91]:
xgb_cv_results = cross_validate(xgb_pipeline, X_train, y_train, cv = skf, scoring = scoring, return_train_score = True)

print("XGBoost Classifier - 5-fold Cross Validation Results\n")
for metric in scoring:
    train_scores = xgb_cv_results[f"train_{metric}"]
    val_scores = xgb_cv_results[f"test_{metric}"]
    print(f"{metric.upper():10s} | Train: {train_scores.mean():.3f} (+/- {train_scores.std():.3f}) | Val: {val_scores.mean():.3f} (+/- {val_scores.std():.3f})")

XGBoost Classifier - 5-fold Cross Validation Results

ACCURACY   | Train: 0.999 (+/- 0.000) | Val: 0.968 (+/- 0.006)
PRECISION  | Train: 0.993 (+/- 0.003) | Val: 0.898 (+/- 0.021)
RECALL     | Train: 1.000 (+/- 0.000) | Val: 0.917 (+/- 0.014)
F1         | Train: 0.997 (+/- 0.001) | Val: 0.907 (+/- 0.017)
ROC_AUC    | Train: 1.000 (+/- 0.000) | Val: 0.987 (+/- 0.001)


In [86]:
xgb_y_pred = xgb_pipeline.predict(X_test)
xgb_y_proba = xgb_pipeline.predict_proba(X_test)[:,1]

xgb_test_scores = {
    'accuracy' : accuracy_score(y_test, xgb_y_pred),
    'precision' : precision_score(y_test, xgb_y_pred),
    'recall' : recall_score(y_test, xgb_y_pred),
    'f1': f1_score(y_test, xgb_y_pred),
    'roc_auc' : roc_auc_score(y_test, xgb_y_proba)
}

print("XGBoost (unconstrained) — Held-Out Test Set Performance\n")
for metric, score in xgb_test_scores.items():
    print(f"{metric.upper():10s} | Test: {score:.3f}")

print("\n--- Comparison: Cross-Validation (mean) vs Test ---")
for metric in scoring:
    cv_val_mean = xgb_cv_results[f"test_{metric}"].mean()
    print(f"{metric.upper():10s} | Cv Val: {cv_val_mean:.3f} | Test: {xgb_test_scores[metric]:.3f} | Diff: {xgb_test_scores[metric] - cv_val_mean:+.3f}")

print("\nClassification Report:\n")
print(classification_report(y_test, xgb_y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_y_pred))

XGBoost (unconstrained) — Held-Out Test Set Performance

ACCURACY   | Test: 0.988
PRECISION  | Test: 0.949
RECALL     | Test: 0.979
F1         | Test: 0.964
ROC_AUC    | Test: 0.999

--- Comparison: Cross-Validation (mean) vs Test ---
ACCURACY   | Cv Val: 0.968 | Test: 0.988 | Diff: +0.019
PRECISION  | Cv Val: 0.898 | Test: 0.949 | Diff: +0.051
RECALL     | Cv Val: 0.917 | Test: 0.979 | Diff: +0.062
F1         | Cv Val: 0.907 | Test: 0.964 | Diff: +0.056
ROC_AUC    | Cv Val: 0.987 | Test: 0.999 | Diff: +0.012

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       936
           1       0.95      0.98      0.96       190

    accuracy                           0.99      1126
   macro avg       0.97      0.98      0.98      1126
weighted avg       0.99      0.99      0.99      1126

Confusion Matrix:
[[926  10]
 [  4 186]]


In [87]:
print(X_train.columns.tolist())
print('CustomerID' in X_train.columns)

['Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferredOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
False


In [88]:
train_test_overlap = pd.merge(X_train, X_test, how='inner')
print("Overlapping rows between train/test:", len(train_test_overlap))

Overlapping rows between train/test: 197


In [89]:
# Identify which test rows have an exact match in training
merge_check = X_test.merge(X_train.drop_duplicates(), how='left', indicator=True)
is_overlap = (merge_check['_merge'] == 'both').values

print("Test rows with exact match in training:", is_overlap.sum())
print("Test rows without a match (genuinely unseen):", (~is_overlap).sum())

# Evaluate performance ONLY on the non-overlapping (genuinely unseen) rows
X_test_clean = X_test[~is_overlap]
y_test_clean = y_test[~is_overlap]

y_pred_clean = xgb_pipeline.predict(X_test_clean)
y_proba_clean = xgb_pipeline.predict_proba(X_test_clean)[:, 1]

print("\n--- XGBoost Performance on GENUINELY UNSEEN test rows only ---")
print(f"Accuracy:  {accuracy_score(y_test_clean, y_pred_clean):.3f}")
print(f"Precision: {precision_score(y_test_clean, y_pred_clean):.3f}")
print(f"Recall:    {recall_score(y_test_clean, y_pred_clean):.3f}")
print(f"F1:        {f1_score(y_test_clean, y_pred_clean):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_clean, y_proba_clean):.3f}")

Test rows with exact match in training: 197
Test rows without a match (genuinely unseen): 929

--- XGBoost Performance on GENUINELY UNSEEN test rows only ---
Accuracy:  0.986
Precision: 0.942
Recall:    0.973
F1:        0.957
ROC-AUC:   0.999


In [94]:
xgb_pipeline_v2 = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        min_child_weight=3,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.05,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    ))
])

In [95]:
xgb_cv_results_v2 = cross_validate(
    xgb_pipeline_v2, X_train, y_train,
    cv=skf, scoring=scoring, return_train_score=True
)

print("XGBoost (v2, moderate constraints) — 5-Fold Cross-Validation Results\n")
for metric in scoring:
    train_scores = xgb_cv_results_v2[f'train_{metric}']
    val_scores = xgb_cv_results_v2[f'test_{metric}']
    print(f"{metric.upper():10s} | Train: {train_scores.mean():.3f} (+/- {train_scores.std():.3f})"
          f"  | Val: {val_scores.mean():.3f} (+/- {val_scores.std():.3f})")

XGBoost (v2, moderate constraints) — 5-Fold Cross-Validation Results

ACCURACY   | Train: 0.985 (+/- 0.001)  | Val: 0.952 (+/- 0.005)
PRECISION  | Train: 0.919 (+/- 0.007)  | Val: 0.820 (+/- 0.020)
RECALL     | Train: 1.000 (+/- 0.000)  | Val: 0.918 (+/- 0.019)
F1         | Train: 0.958 (+/- 0.004)  | Val: 0.866 (+/- 0.015)
ROC_AUC    | Train: 0.999 (+/- 0.000)  | Val: 0.982 (+/- 0.002)


In [96]:
xgb_pipeline_v2.fit(X_train, y_train)

X_test_clean = X_test[~is_overlap]
y_test_clean = y_test[~is_overlap]

y_pred_v2_clean = xgb_pipeline_v2.predict(X_test_clean)
y_proba_v2_clean = xgb_pipeline_v2.predict_proba(X_test_clean)[:, 1]

print("--- XGBoost (v2, moderate) — Genuinely Unseen Test Rows Only ---")
print(f"Accuracy:  {accuracy_score(y_test_clean, y_pred_v2_clean):.3f}")
print(f"Precision: {precision_score(y_test_clean, y_pred_v2_clean):.3f}")
print(f"Recall:    {recall_score(y_test_clean, y_pred_v2_clean):.3f}")
print(f"F1:        {f1_score(y_test_clean, y_pred_v2_clean):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_clean, y_proba_v2_clean):.3f}")

--- XGBoost (v2, moderate) — Genuinely Unseen Test Rows Only ---
Accuracy:  0.969
Precision: 0.862
Recall:    0.960
F1:        0.909
ROC-AUC:   0.991


### Both the XGBClassifier we had trained had the high train - val gap in the cv_results which might be the indicator for the overfitting. So further testing is done to find the best iteration(n_estimator) to prevent the overfitting.

### This is done to find the best_iteration rather than guessing the n_estimator manually

In [108]:
# Further split X_train into a smaller train set + a validation set for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=42
)

# Fit the preprocessor on X_tr only, then transform both X_tr and X_val
tree_preprocessor.fit(X_tr)
X_tr_transformed = tree_preprocessor.transform(X_tr)
X_val_transformed = tree_preprocessor.transform(X_val)

# Recompute scale_pos_weight based on this smaller training split
scale_pos_weight_es = (y_tr == 0).sum() / (y_tr == 1).sum()

xgb_early_stop = XGBClassifier(
    n_estimators=1000,           # set high — early stopping will cut it short
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_es,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42
)

xgb_early_stop.fit(
    X_tr_transformed, y_tr,
    eval_set=[(X_val_transformed, y_val)],
    verbose=False
)

best_n_estimators = xgb_early_stop.best_iteration
print("Best iteration (number of trees actually used):", xgb_early_stop.best_iteration)

Best iteration (number of trees actually used): 263


In [101]:
X_test_clean_transformed = tree_preprocessor.transform(X_test_clean)

y_pred_es = xgb_early_stop.predict(X_test_clean_transformed)
y_proba_es = xgb_early_stop.predict_proba(X_test_clean_transformed)[:, 1]

print("--- XGBoost (early stopping) — Genuinely Unseen Test Rows Only ---")
print(f"Accuracy:  {accuracy_score(y_test_clean, y_pred_es):.3f}")
print(f"Precision: {precision_score(y_test_clean, y_pred_es):.3f}")
print(f"Recall:    {recall_score(y_test_clean, y_pred_es):.3f}")
print(f"F1:        {f1_score(y_test_clean, y_pred_es):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_clean, y_proba_es):.3f}")

--- XGBoost (early stopping) — Genuinely Unseen Test Rows Only ---
Accuracy:  0.983
Precision: 0.953
Recall:    0.940
F1:        0.946
ROC-AUC:   0.997


In [107]:
y_tr_pred = xgb_early_stop.predict(X_tr_transformed)
y_tr_proba = xgb_early_stop.predict_proba(X_tr_transformed)[:, 1]
y_val_pred = xgb_early_stop.predict(X_val_transformed)
y_val_proba = xgb_early_stop.predict_proba(X_val_transformed)[:, 1]

print("--- XGBoost (early stopping) — Full Train vs Val Metrics ---\n")

metrics_train_val = {
    'accuracy': (accuracy_score(y_tr, y_tr_pred), accuracy_score(y_val, y_val_pred)),
    'precision': (precision_score(y_tr, y_tr_pred), precision_score(y_val, y_val_pred)),
    'recall': (recall_score(y_tr, y_tr_pred), recall_score(y_val, y_val_pred)),
    'f1': (f1_score(y_tr, y_tr_pred), f1_score(y_val, y_val_pred)),
    'roc_auc': (roc_auc_score(y_tr, y_tr_proba), roc_auc_score(y_val, y_val_proba))
}

for metric, (train_val, val_val) in metrics_train_val.items():
    gap = train_val - val_val
    print(f"{metric.upper():10s} | Train: {train_val:.3f}  | Val: {val_val:.3f}  | Gap: {gap:.3f}")

--- XGBoost (early stopping) — Full Train vs Val Metrics ---

ACCURACY   | Train: 1.000  | Val: 0.981  | Gap: 0.019
PRECISION  | Train: 0.998  | Val: 0.939  | Gap: 0.059
RECALL     | Train: 1.000  | Val: 0.947  | Gap: 0.053
F1         | Train: 0.999  | Val: 0.943  | Gap: 0.056
ROC_AUC    | Train: 1.000  | Val: 0.994  | Gap: 0.006


### From all the above analysis i get to know that the xgboost classifier comes out to the best model for churn prediction as it has best scoring as compare to logistic regression and random forest classifier.

### And from all the manual tuning and using the early stop, i get to know that the model with early stop gives the best iteration which prevents the overfitting and has less train/val gap as compare to the two other models trained with the xgboost classifier which had high train/val gap. 

### So from all the above model training and evaluation i had chosen the xgboost classifier with the early stop to find the best iteration(n_estimator) to prevent the overfitting.